# **NORMAL PPO** :
**PPO with complete with all the transiton during the rollouts**

In [1]:
!pip install -U gymnasium[mujoco] mujoco

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 48.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 52.6 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.2.0
    Uninstalling gymnasium-1.2.0:
      Successfully uninstalled gymnasium-1.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.27.0 requires gymnasium==1.2.0, but you have gymnasium 1.2.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


# **IMPORTING LIBRARIES**

In [2]:
import torch
import random
import os
import torch.nn as nn
import numpy as np
import time
from collections import deque
import wandb
from tqdm import tqdm
from torch.amp import autocast, GradScaler

import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordVideo

In [3]:
def set_seed(seed):
    import random, numpy as np, torch

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [4]:
def create_environment(cfgs, eval = False):
  env = gym.make( id=cfgs.id)
  return env

# **WANDB RUN**

In [5]:
def wandb_runs(cfg, keep_percent, seed, symbol_no):

  wandb.login(key = "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")
  run = wandb.init(
    entity="ajheshbasnet-kpriet",
    project=f"{cfg.id}-3-15-sunday!!",
    name = f"percentage: {keep_percent*100}%-seed: {seed}-{cfg.id}-symbolno: {symbol_no}",
    config=vars(cfg),
  )

  return run

# **CONFIGURATIONS**

### 


In [6]:
from dataclasses import dataclass

@dataclass
class configuration:
  id = "Hopper-v5"
  max_training_step = 1_200_000
  rollout_steps = 2048
  eval_steps = 12_000
  global_steps = 0
  epochs = [10, 10, 8, 6]
  eval_loops = 3
  batch_size = 256
  d_model = [128, 128, 96, 96]
  ppo_r_clamp = [0.18, 0.18, 0.18, 0.15]
  keep_percent = [1.0, 0.85, 0.75, 0.65]
  critic_lr = [3e-4, 3e-4, 3e-4, 3e-4]
  max_grad_norm = 0.5
  actor_max_grad_norm = 0.5
  critic_max_grad_norm = 0.8
  actor_lr = 9e-5
  record_video = 1e9
  wandblog_step = 100
  device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = configuration()

**Checking environment is working or not:)**

In [7]:
envs = create_environment(cfg)
envs.reset()[0]

array([ 1.25225493e+00,  6.30675165e-05,  2.56053030e-03, -1.50989536e-03,
       -4.93278334e-03,  8.94416506e-04,  4.30313677e-03,  1.55189354e-03,
       -3.65547310e-03, -4.79159474e-03, -3.25167275e-03])

# **Actor and Critic Netowrk**

In [8]:
class Actor(nn.Module):

    def __init__(self, input_dim, action_dim, idxx):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, cfg.d_model[idxx]),
            nn.ReLU(),
            nn.Linear(cfg.d_model[idxx], cfg.d_model[idxx]),
            nn.ReLU(),
            nn.Linear(cfg.d_model[idxx], cfg.d_model[idxx]),
            nn.ReLU(),
        )

        self.mean = nn.Linear(cfg.d_model[idxx], action_dim)

        # global log std parameter
        self.log_std = nn.Parameter(torch.ones(action_dim)*-0.5)


    def forward(self, x):

        x = self.net(x)

        mean = self.mean(x)

        log_std = torch.clamp(self.log_std, -2, 2)

        std = torch.exp(log_std)

        return mean, std


    # =========================
    # SAMPLE ACTION (ROLLOUT)
    # =========================
    def sample_action(self, x):

        mean, std = self(x)

        dist = torch.distributions.Normal(mean, std)

        raw_action = dist.rsample()

        action = torch.tanh(raw_action)

        log_prob = dist.log_prob(raw_action).sum(-1)

        log_prob = log_prob - torch.log(1 - action.pow(2) + 1e-6).sum(-1)

        entropy = dist.entropy().sum(-1).mean()

        return raw_action, action, log_prob, entropy

    # =========================
    # LOG PROB (PPO UPDATE)
    # =========================
    def log_prob(self, states, raw_actions):

        mean, std = self(states)

        dist = torch.distributions.Normal(mean, std)

        log_prob = dist.log_prob(raw_actions).sum(-1)

        action = torch.tanh(raw_actions)

        log_prob = log_prob - torch.log(1 - action.pow(2) + 1e-6).sum(-1)

        entropy = dist.entropy().sum(-1).mean()

        return log_prob, entropy

In [9]:
class Critic(nn.Module):

  def __init__(self, input_dim, idxx):
    super().__init__()

    self.sequential = nn.Sequential(
        nn.Linear(input_dim, cfg.d_model[idxx]),
        nn.ReLU(),
        nn.Linear(cfg.d_model[idxx], cfg.d_model[idxx]),
        nn.ReLU(),
        nn.Linear(cfg.d_model[idxx], 1)
    )

  def forward(self, x):
    x = self.sequential(x)
    return x

In [10]:
actornet = Actor(
    envs.observation_space.shape[0],
    envs.action_space.shape[0], 0
).to(cfg.device)
criticnet = Critic(envs.observation_space.shape[0], 0).to(cfg.device)  #type: ignore

In [11]:
print(f'''Parameters:
==============================
actor-network     : {sum(p.numel() for p in actornet.parameters())/1e3} k
critic-network(s) : {sum(p.numel() for p in criticnet.parameters())/ 1e3} k
==============================
      ''')

Parameters:
actor-network     : 34.95 k
critic-network(s) : 18.177 k
      


In [12]:
os.environ["MUJOCO_GL"] = "egl"

**Evaluation Loop**

In [13]:
def evaluation(actornet, record_video=False):

    eval_env = gym.make(id=cfg.id)
    if record_video:
        eval_env = gym.make(id=cfg.id, render_mode='rgb_array')
        video_dir = f"/kaggle/working/{int(time.time())}"
        eval_env = RecordVideo(eval_env, video_folder=video_dir, episode_trigger=lambda ep: True)

    net_reward = 0
    net_step   = 0

    max_evall_reward = []

    with torch.no_grad():

        for _ in range(cfg.eval_loops):

            done = False
            episodic_reward = 0
            episodic_step   = 0
            state = eval_env.reset()[0]

            while not done:

                stateT = torch.tensor(state, dtype=torch.float32, device=cfg.device)

                with autocast(device_type=cfg.device, dtype=torch.float16, enabled=(cfg.device == "cuda")):
                    mean, std = actornet(stateT)

                # Deterministic eval: use mean, but MUST apply tanh to match training
                action = torch.tanh(mean)

                nxt_state, reward, terminated, truncated, _ = eval_env.step(action.cpu().numpy())

                done   = terminated or truncated
                state  = nxt_state

                episodic_reward += float(reward)
                episodic_step   += 1
            # print(episodic_reward)
            max_evall_reward.append(episodic_reward)
            net_reward += episodic_reward
            net_step   += episodic_step

    net_reward = net_reward / cfg.eval_loops
    net_step   = net_step   / cfg.eval_loops

    eval_env.close()

    return net_reward, net_step, max(max_evall_reward)

In [14]:
evaluation(actornet, False)

(163.75286515392415, 136.0, 172.3441630196911)

**To sample the batches**

**W&B RUNS TO LOG THE METRICS**

In [15]:
gamma = 0.995                                          # for reward decay
lambda_ = 0.96                                         # for GAE tuning
initial_entropy_beta = 1e-4                           # entropy contribution

In [16]:
all_seeds = [0, 42, 123, 2023]

In [ ]:
# ================================
# COMPLETE PPO TRAINING LOOP
# ================================

for idxxx, keep_percent in enumerate(cfg.keep_percent):
        
    for seed in all_seeds:

        symbol_no = torch.randn(1, device=cfg.device) + torch.randn(1, device=cfg.device)
    
        runs = wandb_runs(cfg, keep_percent, seed, symbol_no.item())
    
        envs = create_environment(cfg)
    
        set_seed(seed)
    
        cfg.global_steps = 0
    
        actornet  = Actor(envs.observation_space.shape[0], envs.action_space.shape[0], idxxx).to(cfg.device)
        criticnet = Critic(envs.observation_space.shape[0], idxxx).to(cfg.device)
    
        actor_optimizer  = torch.optim.Adam(actornet.parameters(),  lr=cfg.actor_lr)
        critic_optimizer = torch.optim.Adam(criticnet.parameters(), lr=cfg.critic_lr[idxxx])
    
        scaler_actor  = GradScaler(enabled=(cfg.device == "cuda"))
        scaler_critic = GradScaler(enabled=(cfg.device == "cuda"))
    
        state = envs.reset(seed=seed)[0]
    
        rollout = 1
    
        while cfg.max_training_step > cfg.global_steps:
    
            states      = []
            raw_actions = []
            actions     = []
            next_states = []
            rewards     = []
            log_probs   = []
            dones       = []
    
            training_reward = 0
            rollout_step    = 0
    
            stateT = torch.tensor(state, dtype=torch.float32, device=cfg.device)
    
            # ================= ROLLOUT COLLECTION =================
    
            while rollout_step < cfg.rollout_steps and cfg.global_steps < cfg.max_training_step:
    
                with torch.no_grad():
                    with autocast(device_type=cfg.device, dtype=torch.float16, enabled=(cfg.device=="cuda")):
                        raw_action, action, logprob, _ = actornet.sample_action(stateT)
    
                next_state, reward, terminated, truncated, _ = envs.step(action.cpu().numpy())
    
                done = terminated or truncated
    
                next_stateT = torch.tensor(next_state, dtype=torch.float32, device=cfg.device)
                rewardT = torch.tensor(reward, dtype=torch.float32, device=cfg.device)
                doneT = torch.tensor(done, dtype=torch.float32, device=cfg.device)
    
                states.append(stateT)
                raw_actions.append(raw_action)
                actions.append(action)
                next_states.append(next_stateT)
                rewards.append(rewardT)
                dones.append(doneT)
                log_probs.append(logprob)
    
                stateT = next_stateT
    
                training_reward += float(reward)
                rollout_step += 1
                cfg.global_steps += 1
    
                if cfg.global_steps % cfg.eval_steps == 0:
                    net_reward, net_step, max_eval_reward = evaluation(actornet, False)
                    runs.log({
                        "eval-reward": net_reward,
                        "eval-length": net_step,
                        "max-reward": max_eval_reward
                    }, step=cfg.global_steps)
    
                if done:
                    state = envs.reset()[0]
                    stateT = torch.tensor(state, dtype=torch.float32, device=cfg.device)
                else:
                    state = next_state
    
    
            # ================= STACK ROLLOUT =================
    
            all_states = torch.stack(states)
            all_raw_actions = torch.stack(raw_actions)
            all_actions = torch.stack(actions)
            all_next_states = torch.stack(next_states)
    
            all_rewards = torch.stack(rewards).view(-1,1)
            all_dones = torch.stack(dones).view(-1,1)
            old_log_probs = torch.stack(log_probs).view(-1,1)
    
    
            # ================= GAE ADVANTAGE =================
    
            with torch.no_grad():
    
                values = criticnet(all_states).view(-1,1)
    
                if all_dones[-1]:
                    next_value = torch.zeros((1,1), device=cfg.device)
                else:
                    next_value = criticnet(all_next_states[-1]).view(1,1)
    
            T = all_rewards.size(0)
    
            advantages = torch.zeros_like(all_rewards)
            gae = 0
    
            for t in reversed(range(T)):
    
                next_val = next_value if t == T-1 else values[t+1]
    
                delta = all_rewards[t] + gamma * (1 - all_dones[t]) * next_val - values[t]
    
                gae = delta + gamma * lambda_ * (1 - all_dones[t]) * gae
    
                advantages[t] = gae
    
            returns = advantages + values
    
            advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)
    
            value_bias = (returns - values).mean().item()
    
            explained_var = 1 - ((returns - values).var() / (returns.var() + 1e-8))
    
    
            # ================= PPO UPDATE =================
    
            log_actor_grad_norm  = 0
            log_critic_grad_norm = 0
            kl_divergence        = 0
            log_actor_loss       = 0
            log_policy_loss      = 0
            log_critic_loss      = 0
            log_entropy          = 0
            log_std_mean         = 0
            step = 0

            max_bound = max(1, int(keep_percent*all_states.size(0)))
            shuffled_ids = torch.randperm(all_states.size(0), device=cfg.device)[:max_bound]
    
            shuffled_states = all_states[shuffled_ids]
            shuffled_actions = all_raw_actions[shuffled_ids].squeeze(1)
            shuffled_advantg = advantages[shuffled_ids].squeeze(-1)
            shuffled_returns = returns[shuffled_ids].squeeze(-1)
            shuffled_old_log_probs = old_log_probs[shuffled_ids].squeeze(-1)
            
            batch_size = min(cfg.batch_size, shuffled_states.size(0))

            ppo_clamp = cfg.ppo_r_clamp[idxxx]

            entropy_beta = initial_entropy_beta * (1 - (cfg.global_steps/cfg.max_training_step))
        
            for epoch in range(cfg.epochs[idxxx]):

                new_shuffled_ids = torch.randperm(shuffled_states.size(0), device=cfg.device)

                for i in range(0, shuffled_states.size(0), batch_size):
    
                    batch_ids = new_shuffled_ids[i:i+batch_size]
    
                    mb_states = shuffled_states[batch_ids]
                    mb_raw_actions = shuffled_actions[batch_ids]
    
                    mb_advantages = shuffled_advantg[batch_ids]
                    mb_returns = shuffled_returns[batch_ids]
                    mb_old_log_probs = shuffled_old_log_probs[batch_ids]
    
                    with autocast(device_type=cfg.device, dtype=torch.float16, enabled=(cfg.device=="cuda")):
    
                        mb_new_log_probs, entropy = actornet.log_prob(mb_states, mb_raw_actions)
    
                        ratio = torch.exp(mb_new_log_probs - mb_old_log_probs)
    
                        policy_loss_ = -torch.min(
                            ratio * mb_advantages,
                            torch.clamp(ratio, 1 - ppo_clamp, 1 + ppo_clamp) * mb_advantages
                        ).mean()
    
                        policy_loss = policy_loss_ - entropy_beta * entropy
    
                        critic_loss = torch.nn.functional.mse_loss(
                            criticnet(mb_states).squeeze(-1),
                            mb_returns
                        )
    
                    kl_divergence += (mb_old_log_probs - mb_new_log_probs.float()).mean().item()
    
                    log_actor_loss  += policy_loss.item()
                    log_critic_loss += critic_loss.item()
                    log_entropy     += entropy.item()
                    log_policy_loss += policy_loss_.item()
    
                    log_std_mean += actornet.log_std.exp().mean().item()
    
                    step += 1
    
    
                    # ===== ACTOR UPDATE =====
    
                    actor_optimizer.zero_grad()
    
                    if cfg.device == "cuda":
    
                        scaler_actor.scale(policy_loss).backward()
    
                        scaler_actor.unscale_(actor_optimizer)
    
                        actor_grad_norm = torch.nn.utils.clip_grad_norm_(
                            actornet.parameters(), cfg.max_grad_norm
                        )
    
                        scaler_actor.step(actor_optimizer)
    
                        scaler_actor.update()
    
                    else:
    
                        policy_loss.backward()
    
                        actor_grad_norm = torch.nn.utils.clip_grad_norm_(
                            actornet.parameters(), cfg.actor_max_grad_norm
                        )
    
                        actor_optimizer.step()
    
    
                    # ===== CRITIC UPDATE =====
    
                    critic_optimizer.zero_grad()
    
                    if cfg.device == "cuda":
    
                        scaler_critic.scale(critic_loss).backward()
    
                        scaler_critic.unscale_(critic_optimizer)
    
                        critic_grad_norm = torch.nn.utils.clip_grad_norm_(
                            criticnet.parameters(), cfg.critic_max_grad_norm
                        )
    
                        scaler_critic.step(critic_optimizer)
    
                        scaler_critic.update()
    
                    else:
    
                        critic_loss.backward()
    
                        critic_grad_norm = torch.nn.utils.clip_grad_norm_(
                            criticnet.parameters(), cfg.max_grad_norm
                        )
    
                        critic_optimizer.step()
    
                    log_actor_grad_norm  += actor_grad_norm.item()
                    log_critic_grad_norm += critic_grad_norm.item()
    
    
            # ================= LOGGING =================
    
            runs.log({
    
                "env-number": rollout,
    
                "training-reward": training_reward,
    
                "advantage-mean": advantages.mean().item(),
                "entropy_beta": entropy_beta,
    
                "actor-gradient-norm": log_actor_grad_norm / step,
                "critic-gradient-norm": log_critic_grad_norm / step,
    
                "policy-loss": log_policy_loss / step,
                "actor-loss": log_actor_loss / step,
                "critic-loss": log_critic_loss / step,
    
                "entropy": log_entropy / step,
    
                "kl-divergence": kl_divergence / step,
    
                "explained-variance": explained_var,
    
                "value-bias": value_bias,
    
                "policy-std": log_std_mean / step,
    
                "global-steps": cfg.global_steps,
                
                "idxx": idxxx,
                "d_model": cfg.d_model[idxxx],
                "lr": cfg.critic_lr[idxxx],
                "symbol-no": symbol_no.item()
    
            }, step=cfg.global_steps)
    
            rollout += 1

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ajheshbasnet (ajheshbasnet-kpriet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


actor-gradient-norm,▁▄▅▆▅▆▄▇▄▇▆▃▆▅▅▆▄▅▆▇▅▆▃▆▇▃▅▄▆▅▄▆▆▅▅▇▅▆██
actor-loss,▅▃▃▄▃▄▇▆▇█▇▇▅▆▃▃▅█▆▆▆▂▂▃▅▆▇█▅▇▄▂▂▂▄▅▃▁▅▁
advantage-mean,▃▆▃▃▅▂█▁▂▄▃▅▅▂▄▅▅▃▃▄▆▁▃▃▂▅▃▃▄▄▃▅▅▃▂▂▃▃▄▃
critic-gradient-norm,▁▁▁▂▂ ▃▃▃▃▃▄▂▂▃▄▃▃▄▄▄▄▄▃▄▄▄▆▄▃▃▃▅▄▄█▄▅▅▄
critic-loss,▂▁▁▁▅▆▅▁▅▄▇▂▅▅▅▇▅▃▆▅▃▂▆▆▄▄▂▇▆▃▅▅▇▇▇▆▄▇█▆
d_model,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
entropy,▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▂▂▃▄▄▄▄▃▄▄▄▄▄▄▅▅▆▆▇▇▇▇█
entropy_beta,███▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▁▁▁▁
env-number,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
eval-length,▁▁▂▆█▇▇▆▆██████████████▁█▅█▄██▆█████████
+12,...


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


actor-gradient-norm,▁▁▁▁▂▂▄▅▆▅▆▄▆▅▆▆▅▅▆▆▇▇█▆▆▆▇▅▅▆▇▇▆▆█▇▇▆█▆
actor-loss,▁▃▃▅▄▆▆▅▆███▄█▄▆▆▇▅▆▅▅▆▅█▄▅▅▄▇▅▁▄▅▅▃▇▄▆▂
advantage-mean,▃▂▅▃▂▄▄▁▃▄▂▅▃▃▄▄▃▄█▂▂▃▄▁▄▂▂▃▂▆▅▄▃▂▃▇▅▃▄▄
critic-gradient-norm,▁▁▂▂▃▃▃▃▃▄▄▃▅▄▄▆▃▃▄▅▄▄▅▄▃▃█▅▄▆▄▄▅▆▄▄▄▅▄▄
critic-loss,▁▁▂▁▃▃▁▅▂▇▂▄▄▅▆▃▅▅▃▃▅▄▆▄▄▄▅▅█▄▆▇▃▁▃▅▅▃▅▄
d_model,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
entropy,▂▁▁▁▁▁▁▂▂▂▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▆▆█
entropy_beta,██▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
env-number,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
eval-length,▁▂▂▇▇█▃▄▆▇▅███▇██████▄██████████████████
+12,...


...

actor-gradient-norm,▁▂▂▃▄▄▄▅▄▅▄▅ ▄▆▄▆▅▄▄▆▆▆▅▇▆▆▅▆▆▆▆█▆██▆▇▆█
actor-loss,▄▆▃▃▃█▃▃▃▂▅▅▅▂▅▄▃▃▁▃▅▃▅▄▅▄▁▃▄▅▅▅▄▅▄▄▃▂▄▃
advantage-mean,▁▆██▄▇▆▆▅▅▆▄▄▄▆▆▆▆▆▆▆▇▅▇▆▇▆▃▅▆▆█▅▅▇▅▆▆▆▄
critic-gradient-norm,▁▁▁▁▂▁▁▂▂▂▂▂▂▂▂▃▃▃▃▂▃▃▃▄▃▃▂▆▄▃▃▅▅▅▅▅▆█▄▅
critic-loss,▁▂▅▃▅▄▃▅▄▅▃▂▆▅▃▄▂▂▁▄▄▂▆▅▆▃▃▆▃▅█▂▃▂▆▅▄█▄▆
d_model,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
entropy,▆▆▇▇▇▆▅▄▄▃▄▃▃▃▃▄▆▇█▇▇▇██▇▆▅▂▂▂▁▂▃▄▃▅▄▄▄▄
entropy_beta,███████▇▇▇▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁
env-number,▁▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇█
eval-length,▁▁▁▂▂▃▂▂▂▂▂▂▂▄███▃█▃▄▄▇▆▆▃▅▅▃▄▄▄▆▅█▇█▇▇▇
+12,...


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


actor-gradient-norm,▁▁▁▂▃▅▆▅▄▆▆▅▆▇▅▇█▆▅▄▄▆▆▆▄▄▇▇▇▄▇▄▇█▇▆▅▅█▇
actor-loss,▃▅▃▁▅▄▅▅▄▅▃▄▃▄▇▃▆▃▄▆▆▄█▂▅▇▆▆▃▂▅▄▇▅▂█▅▆▆▄
advantage-mean,▆▂▄▅▂▆▅▆▄▄▅▃▃▄▁▅▅▂▅▃▄▄▅▅▅▄▄▅▅▄█▅▅▆▅▄█▅▄▂
critic-gradient-norm,▁▁▂▂▂▂▂▃▃▃ ▃█▅▃▅▇▆▃▄▃▅▄▅▃▄▃▆▄▃▃▅▃▆▃▃▃▇▄
critic-loss,▁▃▂▂▂▃▃▃▃▅▄▃▃▃▂▅▄█▄▆▃▁▂▂▄▂▃▁▄▆▄▁▄▂▁▂▃▁▁▃
d_model,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
entropy,▃▃▂▂▂▁▁▁▁▁▂▂▃▃▃▃▃▃▃▄▄▃▃▄▃▃▂▂▃▃▆▆▆▇▆▆▆▇██
entropy_beta,██▇▇▇▇▇▆▆▆▅▅▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
env-number,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
eval-length,▁▁▁▁▁▂▄▂▂▇▃▃▄▃▃▆▄▅▃▆▄▇▇▆█▄▅████████▇▅███
+12,...


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


actor-gradient-norm,▁▁▁▂▂▂▂▃▃▄▄▄▄▄▄▅▅▄▅▅▆▆▄▅▆▅▅██▄▅▇▅█▅▇▅▅▆▇
actor-loss,▂▆▄▂▁▅▆▆▆▇▆▅▆▅▆▅▇▃▆▂▅▁▆▇▂▄▅▅█▃▂▅▄█▇▂▂▄▇█
advantage-mean,▁▃▅▃▂▃▃▆▆▃▃▅▃▅▂▃▃▄▃▃▃▂▅▄▃▂▃▃▂▃▃█▃▃█▃▄▃▃▃
critic-gradient-norm,▁▁▂▁▁▂▃▃▄ ▄▃▅▃▄▇▄▄▅▄▄▄ ▄▄█▃ █▇▃▆▅▄▄▅▅▅█▅
critic-loss,▂▁▃▆▆▅▅▆▅▃▄▂▅▇▇▂▇▂▂▄█▂▃▅▁▄▅▅▄▁▄▅▁▃▇▅▂▆▂▅
d_model,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
entropy,████▇█▅▅▆▅▅▂▂▂▂▃▂▁▁▁▂▂▁▁▂▂▃▄▃▄▄▄▄▅▅▅▅▆▆▅
entropy_beta,████▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁
env-number,▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇██
eval-length,▁▁▁▂▂▂▃▂▃▂▂▂▂▃▃▃▃▃▄▄████▄███████▆███▆█▇█
+12,...
